# Inset Lexicon

1. Gabung data bulanan dan hapus duplikat
2. Siapkan data
3. Case folding
4. Character cleaning
5. Normalisasi kata
6. Stemming
7. Stopwords removal
8. Tokenization

# Mount Drive

# Load Data

In [ ]:
import pandas as pd
import re
from collections import Counter

In [ ]:
dataset = pd.read_excel("/content/drive/MyDrive/Tugas Akhir/inset/monthly_kinerja/kinerja.xlsx")

In [ ]:
dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6722 entries, 0 to 6721
Data columns (total 15 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   conversation_id_str      6722 non-null   int64  
 1   created_at               6722 non-null   object 
 2   favorite_count           6722 non-null   int64  
 3   full_text                6722 non-null   object 
 4   id_str                   6722 non-null   int64  
 5   image_url                1651 non-null   object 
 6   in_reply_to_screen_name  3941 non-null   object 
 7   lang                     6722 non-null   object 
 8   location                 0 non-null      float64
 9   quote_count              6722 non-null   int64  
 10  reply_count              6722 non-null   int64  
 11  retweet_count            6722 non-null   int64  
 12  tweet_url                6722 non-null   object 
 13  user_id_str              6722 non-null   object 
 14  username                

# Prepro

## Hapus Duplikasi

In [ ]:
# Cek duplikat berdasarkan kolom id_str
dup_mask = dataset["id_str"].duplicated(keep=False)
dup_count = int(dup_mask.sum())
dup_unique = int(dataset.loc[dup_mask, "id_str"].nunique())
print(f"Jumlah baris duplikat berdasarkan id_str: {dup_count}")
print(f"Jumlah id_str unik yang duplikat: {dup_unique}")
display(dataset.loc[dup_mask].sort_values("id_str").head(20))

Jumlah baris duplikat berdasarkan id_str: 651
Jumlah id_str unik yang duplikat: 114


,conversation_id_str,created_at,favorite_count,full_text,id_str,image_url,in_reply_to_screen_name,lang,location,quote_count,reply_count,retweet_count,tweet_url,user_id_str,username
121,1675815377182552064,2023-07-03T10:34:55.000Z,0,Poengky Indarti mengaku tidak menyangka tingka...,1675815377182552064,https://pbs.twimg.com/ext_tw_video_thumb/16758...,NaN,in,NaN,0,0,0,https://x.com/warungiyan150/status/16758153771...,1530352659533771008,warungiyan150
136,1675815377182552064,2023-07-03T10:34:55.000Z,0,Poengky Indarti mengaku tidak menyangka tingka...,1675815377182552064,https://pbs.twimg.com/ext_tw_video_thumb/16758...,NaN,in,NaN,0,0,0,https://x.com/warungiyan150/status/16758153771...,1530352659533771008,warungiyan150
91,1675815377182552064,2023-07-03T10:34:55.000Z,0,Poengky Indarti mengaku tidak menyangka tingka...,1675815377182552064,https://pbs.twimg.com/ext_tw_video_thumb/16758...,NaN,in,NaN,0,0,0,https://x.com/warungiyan150/status/16758153771...,1530352659533771008,warungiyan150
106,1675815377182552064,2023-07-03T10:34:55.000Z,0,Poengky Indarti mengaku tidak menyangka tingka...,1675815377182552064,https://pbs.twimg.com/ext_tw_video_thumb/16758...,NaN,in,NaN,0,0,0,https://x.com/warungiyan150/status/16758153771...,1530352659533771008,warungiyan150
90,1675877721644350976,2023-07-03T14:42:39.000Z,4,"Tahun 2012, tas ketinggalan di kereta. Kereta ...",1675877721644350976,NaN,NaN,in,NaN,0,2,0,https://x.com/bregmabiyasa/status/167587772164...,120451208,bregmabiyasa
120,1675877721644350976,2023-07-03T14:42:39.000Z,4,"Tahun 2012, tas ketinggalan di kereta. Kereta ...",1675877721644350976,NaN,NaN,in,NaN,0,2,0,https://x.com/bregmabiyasa/status/167587772164...,120451208,bregmabiyasa
105,1675877721644350976,2023-07-03T14:42:39.000Z,4,"Tahun 2012, tas ketinggalan di kereta. Kereta ...",1675877721644350976,NaN,NaN,in,NaN,0,2,0,https://x.com/bregmabiyasa/status/167587772164...,120451208,bregmabiyasa
135,1675877721644350976,2023-07-03T14:42:39.000Z,4,"Tahun 2012, tas ketinggalan di kereta. Kereta ...",1675877721644350976,NaN,NaN,in,NaN,0,2,0,https://x.com/bregmabiyasa/status/167587772164...,120451208,bregmabiyasa
16,1675877721644350976,2023-07-03T14:42:39.000Z,4,"Tahun 2012, tas ketinggalan di kereta. Kereta ...",1675877721644350976,NaN,NaN,in,NaN,0,2,0,https://x.com/bregmabiyasa/status/167587772164...,120451208,bregmabiyasa
104,1675877902200741888,2023-07-03T14:43:22.000Z,0,#KerjaNyataPolri Persentase Publik Trust Polri...,1675877902200741888,https://pbs.twimg.com/media/F0HqEiRaEAM30cT.jpg,NaN,in,NaN,0,0,0,https://x.com/MeliauPolsek/status/167587790220...,1181242559936359936,MeliauPolsek


In [ ]:
dataset.drop_duplicates(subset="id_str", inplace=True)

In [ ]:
# 1. Ambil kolom full_text dan created_at lalu copy ke df
data = dataset[['full_text','created_at', 'id_str', 'lang']].copy()

# 2. Rename kolom di dalam df (bukan di data)
data = data.rename(columns={'full_text': 'text'})

# 3. Lakukan fillna pada df['text'], bukan dataset['text']
data['text'] = data['text'].fillna('').astype(str)

# 4. Ambil tanggal saja dari created_at (hapus jam)
data['created_at'] = pd.to_datetime(data['created_at'], errors='coerce').dt.date.astype(str)

In [ ]:
data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 6185 entries, 0 to 6721
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   text        6185 non-null   object
 1   created_at  6185 non-null   object
 2   id_str      6185 non-null   int64 
 3   lang        6185 non-null   object
dtypes: int64(1), object(3)
memory usage: 241.6+ KB


In [ ]:
data["lang"].value_counts()

,count
lang,
in,6135
zxx,39
qme,10
zh,1


In [ ]:
lang_drop = ["zxx", "qme", "zh"]

data = data[~data["lang"].isin(lang_drop)]

In [ ]:
data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 6135 entries, 0 to 6721
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   text        6135 non-null   object
 1   created_at  6135 non-null   object
 2   id_str      6135 non-null   int64 
 3   lang        6135 non-null   object
dtypes: int64(1), object(3)
memory usage: 239.6+ KB


##  Case Folding

In [ ]:
data['text_casefold'] = data['text'].str.lower()
display(data[['text', 'text_casefold', 'id_str']].head())

,text,text_casefold,id_str
0,Polisi tenan i kon niru kinerja ne polisi sken...,polisi tenan i kon niru kinerja ne polisi sken...,1676278088077046016
1,Persentase Publik Trust Polri Meningkat Survei...,persentase publik trust polri meningkat survei...,1676211295182541056
2,Semoga aplikasi pengaduan ini bisa menyempurna...,semoga aplikasi pengaduan ini bisa menyempurna...,1676177923429256960
3,Semoga aplikasi pengaduan ini bisa menyempurna...,semoga aplikasi pengaduan ini bisa menyempurna...,1676176799817170944
4,Semoga aplikasi pengaduan ini bisa menyempurna...,semoga aplikasi pengaduan ini bisa menyempurna...,1676171300199632896


##  Character Cleaning

In [ ]:
import re
import string

def character_cleaning(text):
    # 1. Pastikan input adalah string
    text = str(text)

    # 2. Hapus tab, new line, backslash, dan karakter non-ASCII (emoji, dll)
    text = text.replace('\\t'," ").replace('\\n'," ").replace('\\u'," ").replace('\\',"")
    text = text.encode('ascii', 'replace').decode('ascii')

    # 3. Hapus "RT" (retweet sign)
    text = re.sub(r'\brt\b', '', text)

    # 4. Hapus URL (lengkap maupun tidak lengkap), Mention, dan Hashtag
    # Menggunakan regex gabungan agar lebih efisien
    text = re.sub(r"([@#][A-Za-z0-9_]+)|(\w+:\/\/\S+)", " ", text)
    text = text.replace("http://", " ").replace("https://", " ")

    # 5. Hapus angka
    text = re.sub(r"\d+", "", text)

    # 6. Hapus duplikasi 3 karakter beruntun atau lebih
    text = re.sub(r'([a-zA-Z])\1\1+', r'\1', text)

    # 7. Replace punctuation dengan spasi supaya kata tidak menempel
    punct_table = str.maketrans({p: " " for p in string.punctuation})
    text = text.translate(punct_table)

    # 8. Hapus single character (huruf sendirian seperti 'a', 'y', 's')
    text = re.sub(r"\b[a-zA-Z]\b", "", text)

    # 9. Handle whitespace (hapus spasi berlebih dan trim kanan-kiri)
    text = re.sub('\\s+', ' ', text).strip()

    return text

# Eksekusi: mengambil dari text_casefold, disimpan ke text_char
data['text_char'] = data['text_casefold'].apply(character_cleaning)

# Menampilkan hasil perbandingannya
display(data[['text_casefold', 'text_char', 'id_str']].head(10))

,text_casefold,text_char,id_str
0,polisi tenan i kon niru kinerja ne polisi sken...,polisi tenan kon niru kinerja ne polisi skena ...,1676278088077046016
1,persentase publik trust polri meningkat survei...,persentase publik trust polri meningkat survei...,1676211295182541056
2,semoga aplikasi pengaduan ini bisa menyempurna...,semoga aplikasi pengaduan ini bisa menyempurna...,1676177923429256960
3,semoga aplikasi pengaduan ini bisa menyempurna...,semoga aplikasi pengaduan ini bisa menyempurna...,1676176799817170944
4,semoga aplikasi pengaduan ini bisa menyempurna...,semoga aplikasi pengaduan ini bisa menyempurna...,1676171300199632896
5,@iulilyy @bening_93 semua orang yang pernah ke...,semua orang yang pernah kehilangan motor gak b...,1676167467327160064
6,@septjunior kenapa polisi kita beda sm yg di 8...,kenapa polisi kita beda sm yg di ya kinerja nya,1676153500517150976
7,setelah semua gak diterima beneran gak mau per...,setelah semua gak diterima beneran gak mau per...,1676117351270600960
8,#kerjanyatapolri persentase publik trust polri...,persentase publik trust polri meningkat survei...,1676096525280346112
9,@mazzini_gsp masih menjadi misteri knp polisi2...,masih menjadi misteri knp polisi wakanda suka ...,1676065150217093120


## tokenisasi

In [ ]:
# Tokenization
data["text_token"] = data["text_char"].apply(lambda x: str(x).split())
data[["text_char", "text_token"]].head()

,text_char,text_token
0,polisi tenan kon niru kinerja ne polisi skena ...,"[polisi, tenan, kon, niru, kinerja, ne, polisi..."
1,persentase publik trust polri meningkat survei...,"[persentase, publik, trust, polri, meningkat, ..."
2,semoga aplikasi pengaduan ini bisa menyempurna...,"[semoga, aplikasi, pengaduan, ini, bisa, menye..."
3,semoga aplikasi pengaduan ini bisa menyempurna...,"[semoga, aplikasi, pengaduan, ini, bisa, menye..."
4,semoga aplikasi pengaduan ini bisa menyempurna...,"[semoga, aplikasi, pengaduan, ini, bisa, menye..."


In [ ]:
# Menampilkan seluruh baris yang terduplikasi berdasarkan text_token
duplikat_rows = data[data.duplicated(subset=['text_token'], keep=False)]

# Urutkan biar pasangan duplikatnya berdekatan saat dilihat
duplikat_rows = duplikat_rows.sort_values(by='text_token')

print(f"Total baris duplikat berdasarkan text_token: {len(duplikat_rows)}")
duplikat_rows[['id_str','text', 'text_char']]

Total baris duplikat berdasarkan text_token: 678


,id_str,text,text_char
2466,1805466046595482112,@Miduk17 𝑗𝑢𝑠𝑡𝑟𝑢 𝑖𝑛𝑖 𝑚𝑒𝑛𝑔𝑢𝑎𝑡𝑘𝑎𝑛 𝑑𝑢𝑔𝑎𝑎𝑛 𝑠𝑒𝑝𝑎𝑟𝑢ℎ ...,
4432,1890543643879937024,"𝕂𝕚𝕥𝕒, 𝕞𝕒𝕤𝕪𝕒𝕣𝕒𝕜𝕒𝕥 𝕡𝕒𝕤𝕥𝕚 𝕞𝕖𝕟𝕘𝕒𝕡𝕣𝕖𝕤𝕚𝕒𝕤𝕚 𝕜𝕚𝕟𝕖𝕣𝕛𝕒 ℙ...",
2643,1809191204485239040,@Aryprasetyo85 𝒕𝒂𝒑𝒊 𝒑𝒐𝒍𝒊𝒔𝒊 𝒋𝒂𝒏𝒈𝒂𝒏 𝒔𝒆𝒐𝒍𝒂𝒉 𝒎𝒂𝒍𝒂𝒊...,
5622,1946948326156553984,Akademisi Papua Apresiasi Kinerja Ops Damai Ca...,akademisi papua apresiasi kinerja ops damai ca...
5626,1946410593843621888,Akademisi Papua Apresiasi Kinerja Ops Damai Ca...,akademisi papua apresiasi kinerja ops damai ca...
...,...,...,...
4874,1910644534859517952,Warga menggeruduk Polsek Cikedung lantaran pol...,warga menggeruduk polsek cikedung lantaran pol...
1559,1755372796765765888,"woi polisi, gak ada korelasinya bilang kinerja...",woi polisi gak ada korelasinya bilang kinerja ...
1558,1755372812922171904,"@tvOneNews woi polisi, gak ada korelasinya bil...",woi polisi gak ada korelasinya bilang kinerja ...
4302,1883759257419706880,"Ya, mmg hak rakyat/warga utk menilai kinerja &...",ya mmg hak rakyat warga utk menilai kinerja am...


In [ ]:
data.drop_duplicates(subset=['text_token'], keep=False, inplace=True)

In [ ]:
data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 5457 entries, 0 to 6721
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   text           5457 non-null   object
 1   created_at     5457 non-null   object
 2   id_str         5457 non-null   int64 
 3   lang           5457 non-null   object
 4   text_casefold  5457 non-null   object
 5   text_char      5457 non-null   object
 6   text_token     5457 non-null   object
dtypes: int64(1), object(6)
memory usage: 341.1+ KB


## Normalisasi kata

In [ ]:
!pip install  nltk

In [ ]:
import nltk
nltk.download('words')

[nltk_data] Downloading package words to /root/nltk_data...
[nltk_data]   Unzipping corpora/words.zip.


True

In [ ]:
import pandas as pd
import nltk
from nltk.corpus import words

# 1. Siapkan kamus Bahasa Inggris (seperti sebelumnya)
english_vocab = set(word.lower() for word in words.words())
indonesian_vocab = pd.read_csv(r"/content/drive/MyDrive/Tugas Akhir/inset/stopwords/kamus_kbbi.txt") # Gunakan ini jika belum digabung

# 3. Fungsi yang sudah diperbarui
def find_pure_english_tokens(token_list):
    if not isinstance(token_list, list):
        return []

    english_tokens = []
    for word in token_list:
        if len(word) > 2 and word in english_vocab and word not in indonesian_vocab:
            english_tokens.append(word)

    return english_tokens

# 4. Terapkan ke dataframe
data['english_tokens'] = data['text_token'].apply(find_pure_english_tokens)

# 5. Cek ulang hasilnya
english_words = set([word for sublist in data['english_tokens'] for word in sublist])
print(f"Total kata Inggris bersih: {len(english_words)}")
print("Contoh:", list(english_words)[:30])

Total kata Inggris bersih: 1409
Contoh: ['bobby', 'don', 'repost', 'islam', 'apa', 'pria', 'safe', 'what', 'over', 'distrust', 'situs', 'china', 'multilateral', 'rating', 'cut', 'piker', 'sim', 'key', 'direct', 'babu', 'fix', 'update', 'edit', 'properly', 'saka', 'drug', 'label', 'manager', 'abies', 'good']


In [ ]:
from pathlib import Path
import re

base_dir = Path(r"/content/drive/MyDrive/Tugas Akhir/stopwords")

# ── load kamusalay ────────────────────────────────────────────────────────────
kamusalay_file = base_dir / "kamusalay_v2.txt"
kamusalay_map = {}
if kamusalay_file.exists():
    text = kamusalay_file.read_text(encoding="utf-8", errors="ignore").lstrip("\ufeff")
    pairs = re.findall(r"[\"']([^\"']+)[\"']\s*:\s*[\"']([^\"']+)[\"']", text)
    if not pairs:
        for raw_line in text.splitlines():
            line = raw_line.strip()
            if not line:
                continue
            parts = re.split(r"\t|,|;|:|=", line, maxsplit=1)
            if len(parts) == 2:
                pairs.append((parts[0], parts[1]))
    for slang, formal in pairs:
        slang  = slang.strip().strip('"').strip("'").lower()
        formal = formal.strip().strip('"').strip("'").lower()
        if slang and formal:
            kamusalay_map[slang] = formal

# ----------------------- sinonim mapping (OOV → sinonim di kamus INSET) ---------------
SINONIM_MAPPING = {
    'mengapresiasi':            'penghargaan',
    'apresiasi':                'penghargaan',
    'gercep':                   'sigap',
    'salut':                    'kagum',
    'mempermudah':              'mudah',
    'profesional':              'terampil',
    'profesionalitas':          'terampil',
    'profesionalisme':          'terampil',
    'memperbaiki':              'baik',
    'ketenangan':               'tenang',
    'berantas':                 'lawan',
    'menghargai':               'penghargaan',
    'bravo':                    'mengesankan',
    'impresif':                 'mengesankan',
    'apik':                     'rapi',
    'keren':                    'luar biasa',
    'hormat':                   'santun',
    'berintegritas':            'jujur',
    'tegakkan':                 'adil',
    'humanis':                  'ramah',
    'jos':                      'luar biasa',
    'joss':                     'luar biasa',
    'rispek':                   'penghargaan',
    'kriminalisasi':            'jahat',
    'hopeless':                 'sia-sia',
    'ngibul':                   'bohong',
    'kacau':                    'berantakan',
    'terbalik':                 'salah',
    'arogan':                   'sombong',
    'palsu':                    'bohong',
    'buset':                    'jengkel',
    'rekayasa':                 'curang',
    'bubar':                    'gagal',
    'kasus':                    'masalah',
    'ngawur':                   'salah',
    'tertawakan':               'ejek',
    'payah':                    'buruk',
    'molor':                    'lalai',
    'dipertanyakan':            'meragukan',
    'negatif':                  'buruk',
    'kacung':                   'hina',
    'pelanggar':                'salah',
    'dibiarkan':                'abai',
    'pencitraan':               'bohong',
    'represif':                 'represi',
    'pertanyakan':              'meragukan',
    'guoblok':                  'goblok',
    'keilangan':                'kehilangan',
    'ngakuin':                  'mengaku',
    'bandingkan':               'membandingkan',
    'bandingin':                'membandingkan',
    'no':                       'tidak',
    'heck':                     'sialan',
    'simple':                   'sederhana',
    'privacy':                  'privasi',
    'poll':                     'jajak pendapat',
    'memorial':                 'peringatan',
    'friend':                   'teman',
    'install':                  'instal',
    'pride':                    'kebanggaan',
    'peacefully':               'dengan damai',
    'public':                   'publik',
    'think':                    'pikir',
    'above':                    'atas',
    'hero':                     'pahlawan',
    'good':                     'bagus',
    'some':                     'beberapa',
    'manager':                  'manajer',
    'least':                    'setidaknya',
    'fly':                      'terbang',
    'thank':                    'terima kasih',
    'these':                    'ini',
    'law':                      'hukum',
    'unpopular':                'tidak populer',
    'leader':                   'pemimpin',
    'finally':                  'akhirnya',
    'can':                      'bisa',
    'ratio':                    'rasio',
    'speak':                    'bicara',
    'joke':                     'lelucon',
    'power':                    'kekuatan',
    'really':                   'sangat',
    'still':                    'masih',
    'body':                     'tubuh',
    'point':                    'poin',
    'play':                     'main',
    'request':                  'permintaan',
    'commander':                'komandan',
    'shift':                    'sif',
    'polling':                  'poling',
    'event':                    'acara',
    'consent':                  'persetujuan',
    'series':                   'seri',
    'victim':                   'korban',
    'source':                   'sumber',
    'strong':                   'kuat',
    'about':                    'tentang',
    'move':                     'pindah',
    'tonight':                  'malam ini',
    'expose':                   'bongkar',
    'wish':                     'harap',
    'optimization':             'optimisasi',
    'personally':               'secara pribadi',
    'spy':                      'mata-mata',
    'police':                   'polisi',
    'press':                    'pers',
    'reset':                    'atur ulang',
    'jump':                     'lompat',
    'performance':              'kinerja',
    'project':                  'proyek',
    'detective':                'detektif',
    'retarded':                 'terbelakang',
    'cement':                   'semen',
    'statement':                'pernyataan',
    'based':                    'berdasarkan',
    'must':                     'harus',
    'open':                     'buka',
    'trend':                    'tren',
    'system':                   'sistem',
    'scene':                    'adegan',
    'black':                    'hitam',
    'complain':                 'komplain',
    'reply':                    'balasan',
    'chaos':                    'kekacauan',
    'something':                'sesuatu',
    'experimentation':          'eksperimen',
    'try':                      'coba',
    'opinion':                  'opini',
    'plot':                     'alur',
    'down':                     'bawah',
    'perpetrator':              'pelaku',
    'arc':                      'babak',
    'dont':                     'jangan',
    'take':                     'ambil',
    'now':                      'sekarang',
    'cut':                      'potong',
    'intern':                   'magang',
    'apartment':                'apartemen',
    'cooling':                  'pendingin',
    'comprehensive':            'komprehensif',
    'you':                      'kamu',
    'why':                      'mengapa',
    'center':                   'pusat',
    'coexist':                  'berdampingan',
    'normalize':                'normalisasi',
    'thread':                   'utas',
    'fair':                     'adil',
    'scam':                     'penipuan',
    'professional':             'profesional',
    'tax':                      'pajak',
    'citizen':                  'warga negara',
    'sound':                    'suara',
    'individualism':            'individualisme',
    'best':                     'terbaik',
    'lucky':                    'beruntung',
    'research':                 'penelitian',
    'nation':                   'bangsa',
    'chaser':                   'pengejar',
    'random':                   'acak',
    'foreigner':                'orang asing',
    'agent':                    'agen',
    'love':                     'cinta',
    'fire':                     'api',
    'resign':                   'mengundurkan diri',
    'zero':                     'nol',
    'meeting':                  'rapat',
    'give':                     'beri',
    'bare':                     'kosong',
    'year':                     'tahun',
    'member':                   'anggota',
    'changer':                  'pengubah',
    'core':                     'inti',
    'match':                    'pertandingan',
    'woman':                    'wanita',
    'what':                     'apa',
    'conference':               'konferensi',
    'training':                 'pelatihan',
    'province':                 'provinsi',
    'until':                    'hingga',
    'reward':                   'hadiah',
    'japan':                    'jepang',
    'channel':                  'saluran',
    'trigger':                  'pemicu',
    'judge':                    'hakim',
    'team':                     'tim',
    'glowing':                  'bercahaya',
    'corporate':                'perusahaan',
    'epic':                     'epik',
    'regency':                  'kabupaten',
    'respect':                  'hormat',
    'nothing':                  'tidak ada',
    'letter':                   'surat',
    'technology':               'teknologi',
    'human':                    'manusia',
    'kids':                     'anak-anak',
    'role':                     'peran',
    'far':                      'jauh',
    'slow':                     'lambat',
    'surprisingly':             'secara mengejutkan',
    'bag':                      'tas',
    'justice':                  'keadilan',
    'velocity':                 'kecepatan',
    'hacking':                  'peretasan',
    'spending':                 'pengeluaran',
    'abuse':                    'pelecehan',
    'map':                      'peta',
    'during':                   'selama',
    'urgent':                   'mendesak',
    'deaf':                     'tuli',
    'that':                     'itu',
    'going':                    'pergi',
    'worth':                    'sepadan',
    'creator':                  'pembuat',
    'silent':                   'diam',
    'ranger':                   'penjaga',
    'steal':                    'curi',
    'anything':                 'apa saja',
    'miss':                     'rindu',
    'keep':                     'simpan',
    'uncover':                  'mengungkap',
    'properly':                 'dengan benar',
    'twist':                    'kejutan',
    'trading':                  'perdagangan',
    'chef':                     'koki',
    'group':                    'grup',
    'form':                     'formulir',
    'follow':                   'ikuti',
    'have':                     'punya',
    'big':                      'besar',
    'seriously':                'dengan serius',
    'election':                 'pemilihan',
    'key':                      'kunci',
    'luck':                     'keberuntungan',
    'ran':                      'lari',
    'programmer':               'pemrogram',
    'people':                   'orang-orang',
    'comment':                  'komentar',
    'translate':                'terjemahkan',
    'speed':                    'kecepatan',
    'corps':                    'korps',
    'pad':                      'bantalan',
    'building':                 'bangunan',
    'template':                 'templat',
    'piercing':                 'tindik',
    'banner':                   'spanduk',
    'please':                   'tolong',
    'hip':                      'panggul',
    'issue':                    'masalah',
    'china':                    'tiongkok',
    'meanwhile':                'sementara itu',
    'bully':                    'rundung',
    'know':                     'tahu',
    'hard':                     'keras',
    'text':                     'teks',
    'septic':                   'septik',
    'amen':                     'amin',
    'burn':                     'bakar',
    'impact':                   'dampak',
    'always':                   'selalu',
    'misleading':               'menyesatkan',
    'governance':               'tata kelola',
    'resort':                   'resor',
    'second':                   'kedua',
    'operation':                'operasi',
    'denial':                   'penyangkalan',
    'army':                     'tentara',
    'goes':                     'pergi',
    'rate':                     'tingkat',
    'watch':                    'tonton',
    'warehouse':                'gudang',
    'update':                   'perbarui',
    'trying':                   'coba',
    'image':                    'gambar',
    'feedback':                 'umpan balik',
    'charge':                   'isi daya',
    'line':                     'garis',
    'counter':                  'loket',
    'moment':                   'momen',
    'back':                     'kembali',
    'interpreter':              'penerjemah',
    'error':                    'galat',
    'expect':                   'harap',
    'getting':                  'dapat',
    'background':               'latar belakang',
    'happy':                    'senang',
    'walk':                     'jalan',
    'self':                     'diri',
    'independent':              'independen',
    'problem':                  'masalah',
    'girl':                     'gadis',
    'posted':                   'diunggah',
    'not':                      'tidak',
    'productive':               'produktif',
    'camera':                   'kamera',
    'approve':                  'setuju',
    'yes':                      'ya',
    'indicator':                'indikator',
    'bat':                      'kelelawar',
    'rat':                      'tikus',
    'equality':                 'kesetaraan',
    'annoying':                 'menyebalkan',
    'fast':                     'cepat',
    'another':                  'yang lain',
    'foul':                     'pelanggaran',
    'record':                   'rekam',
    'sea':                      'laut',
    'anyway':                   'omong-omong',
    'action':                   'aksi',
    'bet':                      'taruhan',
    'asset':                    'aset',
    'slide':                    'geser',
    'rock':                     'batu',
    'pure':                     'murni',
    'perform':                  'tampil',
    'refresh':                  'segarkan',
    'number':                   'nomor',
    'hell':                     'neraka',
    'signal':                   'sinyal',
    'pen':                      'pena',
    'slice':                    'irisan',
    'lion':                     'singa',
    'leadership':               'kepemimpinan',
    'officer':                  'petugas',
    'award':                    'penghargaan',
    'for':                      'untuk',
    'review':                   'ulasan',
    'safe':                     'aman',
    'improve':                  'tingkatkan',
    'plan':                     'rencana',
    'time':                     'waktu',
    'external':                 'eksternal',
    'cottage':                  'pondok',
    'supporter':                'pendukung',
    'fix':                      'perbaiki',
    'attitude':                 'sikap',
    'enjoy':                    'nikmati',
    'less':                     'kurang',
    'west':                     'barat',
    'idol':                     'idola',
    'itself':                   'itu sendiri',
    'balance':                  'keseimbangan',
    'jot':                      'catat',
    'support':                  'dukungan',
    'stop':                     'berhenti',
    'content':                  'konten',
    'spill':                    'ungkap',
    'from':                     'dari',
    'chairman':                 'ketua',
    'suspect':                  'tersangka',
    'tweet':                    'cuitan',
    'view':                     'tampilan',
    'top':                      'atas',
    'upgrade':                  'tingkatkan',
    'prime':                    'utama',
    'risky':                    'berisiko',
    'behavior':                 'perilaku',
    'medium':                   'sedang',
    'activity':                 'aktivitas',
    'solve':                    'selesaikan',
    'mate':                     'teman',
    'acknowledge':              'akui',
    'had':                      'telah',
    'horror':                   'horor',
    'industry':                 'industri',
    'everyday':                 'setiap hari',
    'touring':                  'tur',
    'vandal':                   'perusak',
    'level':                    'tingkat',
    'dialogue':                 'dialog',
    'better':                   'lebih baik',
    'naive':                    'naif',
    'dollar':                   'dolar',
    'test':                     'tes',
    'tube':                     'tabung',
    'then':                     'kemudian',
    'sampling':                 'pengambilan sampel',
    'part':                     'bagian',
    'breaking':                 'patah',
    'spot':                     'titik',
    'thinking':                 'berpikir',
    'thumbnail':                'gambar mini',
    'survey':                   'survei',
    'headline':                 'tajuk utama',
    'got':                      'dapat',
    'dislike':                  'tidak suka',
    'head':                     'kepala',
    'owl':                      'burung hantu',
    'accountable':              'akuntabel',
    'actually':                 'sebenarnya',
    'real':                     'nyata',
    'express':                  'ekspres',
    'founder':                  'pendiri',
    'partner':                  'mitra',
    'work':                     'kerja',
    'compare':                  'bandingkan',
    'smart':                    'pintar',
    'bid':                      'tawaran',
    'jar':                      'stoples',
    'done':                     'selesai',
    'edit':                     'sunting',
    'skill':                    'keterampilan',
    'woe':                      'duka',
    'hyper':                    'hiper',
    'news':                     'berita',
    'story':                    'cerita',
    'terrorism':                'terorisme',
    'fit':                      'pas',
    'case':                     'kasus',
    'blow':                     'tiup',
    'scroll':                   'gulir',
    'import':                   'impor',
    'villain':                  'penjahat',
    'zombie':                   'zombi',
    'change':                   'ubah',
    'forfeiture':               'perampasan',
    'deserve':                  'pantas',
    'tone':                     'nada',
    'elf':                      'peri',
    'requirement':              'persyaratan',
    'sex':                      'seks',
    'capable':                  'mampu',
    'familiar':                 'akrab',
    'bodyguard':                'pengawal',
    'gag':                      'lelucon',
    'amaze':                    'kagum',
    'device':                   'perangkat',
    'this':                     'ini',
    'set':                      'atur',
    'backing':                  'dukungan',
    'vote':                     'pilih',
    'check':                    'periksa',
    'roasting':                 'memanggang',
    'rob':                      'rampok',
    'hospital':                 'rumah sakit',
    'should':                   'harus',
    'will':                     'akan',
    'assessment':               'penilaian',
    'care':                     'peduli',
    'mention':                  'sebut',
    'coffee':                   'kopi',
    'hacker':                   'peretas',
    'player':                   'pemain',
    'trial':                    'uji coba',
    'art':                      'seni',
    'net':                      'bersih',
    'collapse':                 'runtuh',
    'reliable':                 'andal',
    'saying':                   'ucapan',
    'criminal':                 'kriminal',
    'commandant':               'komandan',
    'ugly':                     'jelek',
    'punishment':               'hukuman',
    'massive':                  'masif',
    'unsolved':                 'belum terpecahkan',
    'list':                     'daftar',
    'claim':                    'klaim',
    'scare':                    'menakut-nakuti',
    'lazy':                     'malas',
    'blame':                    'salah',
    'footage':                  'cuplikan',
    'repost':                   'unggah ulang',
    'see':                      'lihat',
    'praying':                  'berdoa',
    'influencer':               'pemengaruh',
    'grand':                    'besar',
    'notice':                   'perhatikan',
    'applause':                 'tepuk tangan',
    'dashboard':                'dasbor',
    'quote':                    'kutipan',
    'sorry':                    'maaf',
    'stand':                    'berdiri',
    'other':                    'lain',
    'direct':                   'langsung',
    'gang':                     'geng',
    'link':                     'tautan',
    'distrust':                 'ketidakpercayaan',
    'traffic':                  'lalu lintas',
    'proper':                   'layak',
    'TRUE':                     'benar',
    'towel':                    'handuk',
    'track':                    'lintasan',
    'one':                      'satu',
    'two':                      'dua',
    'highlight':                'sorotan',
    'boy':                      'anak laki-laki',
    'december':                 'desember',
    'circle':                   'lingkaran',
    'dispatch':                 'pengiriman',
    'engagement':               'keterlibatan',
    'pick':                     'pilih',
    'massif':                   'masif',
    'draft':                    'draf',
    'placement':                'penempatan',
    'well':                     'baik',
    'framing':                  'pembingkaian',
    'study':                    'studi',
    'downgrade':                'turunkan',
    'rule':                     'aturan',
    'leak':                     'bocor',
    'intended':                 'dimaksudkan',
    'world':                    'dunia',
    'gate':                     'gerbang',
    'official':                 'resmi',
    'fact':                     'fakta',
    'low':                      'rendah',
    'they':                     'mereka',
    'double':                   'ganda',
    'seaman':                   'pelaut',
    'worn':                     'usang',
    'beyond':                   'melampaui',
    'guess':                    'tebak',
    'boss':                     'bos',
    'opus':                     'karya',
    'act':                      'tindak',
    'minority':                 'minoritas',
    'brother':                  'saudara',
    'breakout':                 'terobosan',
    'useless':                  'tidak berguna',
    'transparency':             'transparansi',
    'personal':                 'pribadi',
    'morning':                  'pagi',
    'smoker':                   'perokok',
    'great':                    'hebat',
    'hub':                      'pusat',
    'even':                     'bahkan',
    'syndrome':                 'sindrom',
    'reasonable':               'masuk akal',
    'works':                    'bekerja',
    'gentle':                   'lembut',
    'dear':                     'sayang',
    'call':                     'panggil',
    'ticketing':                'peniketan',
    'show':                     'pertunjukan',
    'ending':                   'akhir',
    'life':                     'hidup',
    'driver':                   'sopir',
    'crime':                    'kejahatan',
    'negative':                 'negatif',
    'war':                      'perang',
    'satan':                    'setan',
    'shoot':                    'tembak',
    'said':                     'berkata',
    'money':                    'uang',
    'attack':                   'serang',
    'over':                     'atas',
    'powerful':                 'kuat',
    'store':                    'toko',
    'ballroom':                 'ruang dansa',
    'wait':                     'tunggu',
    'because':                  'karena',
    'wake':                     'bangun',
    'full':                     'penuh',
    'tea':                      'teh',
    'instead':                  'sebagai gantinya',
    'like':                     'seperti',
    'host':                     'tuan rumah',
    'blind':                    'buta',
    'gimmick':                  'gimik',
    'skip':                     'lewati',
    'ass':                      'pantat',
    'amazed':                   'takjub',
    'appreciate':               'hargai',
    'mostly':                   'sebagian besar',
    'control':                  'kontrol',
    'drug':                     'obat',
    'landscape':                'lanskap',
    'drop':                     'jatuh',
    'account':                  'akun',
    'standard':                 'standar',
    'savage':                   'buas',
    'shut':                     'tutup',
    'government':               'pemerintah',
    'again':                    'lagi',
    'comedy':                   'komedi',
    'auto':                     'otomatis',
    'shock':                    'kejut',
    'tag':                      'tandai',
    'share':                    'bagikan',
    'protect':                  'lindungi',
    'park':                     'taman',
    'trust':                    'kepercayaan',
    'scan':                     'pindai',
    'island':                   'pulau',
    'prank':                    'kelakar',
    'step':                     'langkah',
    'incest':                   'inses',
    'say':                      'katakan',
    'privilege':                'hak istimewa',
    'relate':                   'berhubungan',
    'notebook':                 'buku catatan',
    'stay':                     'tinggal',
    'voter':                    'pemilih',
    'lost':                     'hilang',
    'screen':                   'layar',
    'cause':                    'sebab',
    'bail':                     'jaminan',
    'hate':                     'benci',
    'publish':                  'terbitkan',
    'instant':                  'instan',
    'credit':                   'kredit',
    'make':                     'buat',
    'community':                'komunitas',
    'fully':                    'sepenuhnya',
    'hear':                     'dengar',
    'millions':                 'jutaan',
    'nice':                     'bagus',
    'order':                    'pesanan',
    'product':                  'produk',
    'except':                   'kecuali',
    'bell':                     'bel',
    'paper':                    'kertas',
    'blur':                     'kabur',
    'political':                'politis',
    'security':                 'keamanan',
    'after':                    'setelah',
    'voice':                    'suara',
    'owner':                    'pemilik',
    'staff':                    'staf',
    'union':                    'serikat',
    'progress':                 'kemajuan',
    'exclusive':                'eksklusif',
    'contraflow':               'lawan arus',
    'romance':                  'romansa',
    'judicative':               'yudikatif',
    'lose':                     'kalah',
    'awkward':                  'canggung',
    'angle':                    'sudut',
    'before':                   'sebelum',
    'men':                      'pria',
    'live':                     'langsung',
    'beating':                  'pukulan',
    'fighting':                 'pertarungan',
    'all':                      'semua',
    'index':                    'indeks',
    'foundation':               'yayasan',
    'wonder':                   'heran',
    'apart':                    'terpisah',
    'supporting':               'mendukung',
    'toy':                      'mainan',
    'bold':                     'tebal',
    'game':                     'permainan',
    'shame':                    'rasa malu',
    'hit':                      'pukul',
    'liberalism':               'liberalisme',
    'season':                   'musim',
    'high':                     'tinggi',
    'clear':                    'jelas',
    'contract':                 'kontrak',
    'and':                      'dan',
    'your':                     'milikmu',
    'gold':                     'emas',
    'name':                     'nama',
    'noise':                    'kebisingan',
    'sample':                   'sampel',
    'damned':                   'terkutuk',
    'command':                  'perintah',
    'turmoil':                  'kekacauan',
    'sad':                      'sedih',
    'hopeless':                 'putus asa',
    'livery':                   'corak',
    'sender':                   'pengirim',
    'hoax':                     'hoaks',
    'ate':                      'makan',
    'come':                     'datang',
    'aware':                    'sadar',
    'are':                      'adalah',
    'chat':                     'obrolan',
    'social':                   'sosial',
    'rating':                   'peringkat',
    'out':                      'keluar',
    'original':                 'asli',
    'handle':                   'tangani',
    'same':                     'sama',
    'hint':                     'petunjuk',
    'centrum':                  'pusat',
    'spit':                     'ludah',
    'quality':                  'kualitas',
    'item':                     'barang',
    'piece':                    'potongan',
    'next':                     'selanjutnya',
    'sub':                      'sub',
    'pict':                     'gambar',
    'let':                      'biarkan',
    'superhero':                'pahlawan super',
    'deserved':                 'pantas',
    'international':            'internasional',
    'but':                      'tapi',
    'written':                  'tertulis',
    'humble':                   'rendah hati',
    'bottom':                   'bawah',
    'campaign':                 'kampanye',
    'indonesian':               'orang indonesia',
    'chase':                    'kejar',
    'united':                   'bersatu',
    'speechless':               'tanpa kata',
    'kind':                     'jenis',
    'takedown':                 'turunkan',
    'politicians':              'politisi',
    'awards':                   'penghargaan',
    'banned':                   'dilarang',
    'ndek':                     'di',
    'meh':                      'mau',
    'dicedeki':                 'didekati',
    'ngmonge':                  'ngomongnya',
    'sbntr':                    'sebentar',
    'dijupuk':                  'diambil',
    'onok':                     'ada',
    'maringono':                'setelah itu',
    'dewe':                     'sendiri',
    'matane':                   'matanya',
    'raonok':                   'tidak ada',
    'sing':                     'yang',
    'kelar':                    'selesai',
    'jarene':                   'katanya',
    'digaplok':                 'ditampar',
    'raurus':                   'ratusan',
    'turu':                     'tidur',
    'wae':                      'saja',
    'nyusahke':                 'menyusahkan',
    'opo':                      'apa',
    'meneh':                    'lagi',
    'tangi':                    'bangun',
    'kui':                      'itu',
    'karep':                    'sesukanya',
    'kon':                      'kamu',
    'yen':                      'kalau',
    'tumindakmu':               'kelakuanmu',
    'apik':                     'baik',
    'koyo':                     'seperti',
    'ngen':                     'itu',
    'shgga':                    'sehingga',
    'tenan':                    'harus',
}

# ----------------------- keep slang yang overlap dengan leksikon -----------------------
keep_slang = set()
try:
    keep_slang.update(slang_overlap_pos)
    keep_slang.update(slang_overlap_neg)
except Exception:
    pass

# ----------------------- fungsi normalisasi -------------------------------------------
def normalize_slang(words):
    if words is None:
        return []
    if isinstance(words, list):
        tokens = [str(w).strip().lower() for w in words if str(w).strip()]
    else:
        single = str(words).strip().lower()
        tokens = [single] if single else []

    normalized = []
    for word in tokens:
        # Langkah 1: pertahankan slang yang overlap dengan leksikon
        if word in keep_slang:
            normalized.append(word)
            continue

        # Langkah 2: normalisasi slang → formal via kamusalay
        word = kamusalay_map.get(word, word)

        # Langkah 3: normalisasi OOV → sinonim yang ada di kamus INSET
        word = SINONIM_MAPPING.get(word, word)

        normalized.append(word)
    return normalized

data["text_norm"] = data["text_token"].apply(normalize_slang)
print(data["text_norm"].head())

0    [polisi, harus, kamu, meniru, kinerja, ini, po...
3    [semoga, aplikasi, pengaduan, ini, bisa, menye...
5    [semua, orang, yang, pernah, kehilangan, motor...
6    [kenapa, polisi, kita, beda, sama, yang, di, p...
7    [setelah, semua, enggak, diterima, benaran, en...
Name: text_norm, dtype: object


## Stopwords Removal

In [ ]:
!pip install sastrawi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 3.8 MB/s eta 0:00:00


In [ ]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [ ]:
from pathlib import Path
from nltk.corpus import stopwords
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
import pandas as pd

base_dir = Path(r"/content/drive/MyDrive/Tugas Akhir/inset/stopwords")
lex_pos_path = Path(r"D:\Kuliah\Tugas Akhir\positive.tsv")
lex_neg_path = Path(r"D:\Kuliah\Tugas Akhir\negative.tsv")

# ----------------------- get stopword from NLTK stopword -------------------------------
list_stopwords = stopwords.words("indonesian")
sastrawi_stopwords = StopWordRemoverFactory().get_stop_words()
list_stopwords.extend(sastrawi_stopwords)
# ---------------------------------------------------------------------------------------
# normalize stopwords to match text_char/text_norm
list_stopwords = {str(w).strip().lower() for w in list_stopwords if str(w).strip()}

NEGATION_WORDS = {
    "tidak", "tak", "bukan", "belum", "jangan", "gak",
    "nggak", "ngga", "enggak", "tanpa", "tiada",
}

# remove stopwords that overlap with sentiment lexicons
lex_words = set()
if lex_pos_path.exists():
    lex_pos = pd.read_csv(lex_pos_path, sep="\t", header=None)
    lex_words.update({str(w).strip().lower() for w in lex_pos[0].dropna() if str(w).strip()})
if lex_neg_path.exists():
    lex_neg = pd.read_csv(lex_neg_path, sep="\t", header=None)
    lex_words.update({str(w).strip().lower() for w in lex_neg[0].dropna() if str(w).strip()})

list_stopwords = list_stopwords - (list_stopwords & lex_words)
list_stopwords = list_stopwords - NEGATION_WORDS

# remove stopword pada list token (tanpa stemming)
def stopwords_removal(words):
    if words is None:
        return []
    if isinstance(words, list):
        tokens = [str(w).strip().lower() for w in words if str(w).strip()]
    else:
        tokens = [w.strip().lower() for w in str(words).split() if w.strip()]
    return [word for word in tokens if word not in list_stopwords]

data["text_filtered"] = data["text_norm"].apply(stopwords_removal)
print(data["text_filtered"].head())

0    [polisi, meniru, kinerja, polisi, skena, lho, ...
3    [semoga, aplikasi, pengaduan, menyempurnakan, ...
5    [orang, kehilangan, motor, enggak, mengaku, ki...
6    [polisi, beda, panggilannya, kinerja, orang-or...
7    [enggak, diterima, benaran, enggak, percaya, p...
Name: text_filtered, dtype: object


In [ ]:
data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 5457 entries, 0 to 6721
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   text            5457 non-null   object
 1   created_at      5457 non-null   object
 2   id_str          5457 non-null   int64 
 3   lang            5457 non-null   object
 4   text_casefold   5457 non-null   object
 5   text_char       5457 non-null   object
 6   text_token      5457 non-null   object
 7   english_tokens  5457 non-null   object
 8   text_norm       5457 non-null   object
 9   text_filtered   5457 non-null   object
dtypes: int64(1), object(9)
memory usage: 469.0+ KB


## Save Hasil Prepocessing

In [ ]:
from pathlib import Path

output_dir = Path("preprocessing results")
output_dir.mkdir(parents=True, exist_ok=True)
output_file = output_dir / "kinerja_prepro.xlsx"

data.to_excel(output_file, index=False)
print(f"Hasil disimpan ke: {output_file}")

Hasil disimpan ke: preprocessing results/kinerja_prepro.xlsx


In [ ]:
df_inset = data.drop(columns=['text_casefold','text_char','text_norm','english_tokens','text_token', 'lang'])
df_inset.head()

,text,created_at,id_str,text_filtered
0,Polisi tenan i kon niru kinerja ne polisi sken...,2023-07-04,1676278088077046016,"[polisi, meniru, kinerja, polisi, skena, lho, ..."
3,Semoga aplikasi pengaduan ini bisa menyempurna...,2023-07-04,1676176799817170944,"[semoga, aplikasi, pengaduan, menyempurnakan, ..."
5,@IUlilyy @bening_93 Semua orang yang pernah ke...,2023-07-04,1676167467327160064,"[orang, kehilangan, motor, enggak, mengaku, ki..."
6,@septjunior Kenapa polisi kita beda sm yg di 8...,2023-07-04,1676153500517150976,"[polisi, beda, panggilannya, kinerja, orang-or..."
7,setelah semua gak diterima beneran gak mau per...,2023-07-04,1676117351270600960,"[enggak, diterima, benaran, enggak, percaya, p..."


In [ ]:
df_inset.info()

<class 'pandas.core.frame.DataFrame'>
Index: 5457 entries, 0 to 6721
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   text           5457 non-null   object
 1   created_at     5457 non-null   object
 2   id_str         5457 non-null   int64 
 3   text_filtered  5457 non-null   object
dtypes: int64(1), object(3)
memory usage: 342.2+ KB


# Inset

In [ ]:
import pandas as pd
import ast

# 1. Fungsi untuk memuat kamus (lexicon) Inset
def load_lexicon(path):
    df_lex = pd.read_csv(path, sep="\t", header=None)
    lex = {}
    for _, row in df_lex.iterrows():
        key = str(row[0]).strip().lower()
        if not key:
            continue
        try:
            val = float(row[1])
        except Exception:
            val = 0.0
        lex.setdefault(key, val)
    return lex

# Sesuaikan direktori dengan lokasi file kamus Anda
lexicon_positive_dict = load_lexicon(r"/content/drive/MyDrive/Tugas Akhir/inset/positive.tsv")
lexicon_negative_dict = load_lexicon(r"/content/drive/MyDrive/Tugas Akhir/inset/negative.tsv")

# 2. Aturan Negasi
NEGATION_WORDS = {
    "tidak", "tak", "bukan", "belum", "jangan", "gak",
    "nggak", "ngga", "enggak", "tanpa", "tiada",
}
NEGATION_WINDOW = 2

# Menggabungkan phrase (frasa) dari kamus positif dan negatif
phrase_keys = {
    k for k in set(lexicon_positive_dict) | set(lexicon_negative_dict)
    if " " in k
}

# 3. Fungsi untuk mengambil skor dari tiap token
def get_token_score(token):
    return lexicon_positive_dict.get(token, 0.0) + lexicon_negative_dict.get(token, 0.0)

# 4. Fungsi Utama Analisis Sentimen Inset
def sentiment_analysis_lexicon_indonesia(
    text, negation_words=NEGATION_WORDS, window_size=NEGATION_WINDOW
):
    # Penanganan aman jika input berupa representasi string dari list (misal: "['kata', 'kedua']")
    if isinstance(text, str) and text.startswith('[') and text.endswith(']'):
        try:
            tokens_list = ast.literal_eval(text)
            tokens = [str(t).strip().lower() for t in tokens_list if str(t).strip()]
        except Exception:
            tokens = [t.strip().lower() for t in text.split() if t.strip()]
    elif isinstance(text, list):
        tokens = [str(t).strip().lower() for t in text if str(t).strip()]
    elif text is None:
        tokens = []
    else:
        tokens = [t.strip().lower() for t in str(text).split() if t.strip()]

    score = 0.0
    i = 0
    token_scores = {} # Dictionary untuk menyimpan token beserta kata negasinya (jika ada)

    while i < len(tokens):
        matched_len = 1
        word_score = 0.0
        matched_phrase = ""

        # Cek trigram (3 kata)
        if i + 2 < len(tokens):
            tri = f"{tokens[i]} {tokens[i + 1]} {tokens[i + 2]}"
            if tri in phrase_keys:
                word_score = get_token_score(tri)
                matched_len = 3
                matched_phrase = tri

        # Cek bigram (2 kata) jika trigram tidak cocok
        if word_score == 0.0 and i + 1 < len(tokens):
            bi = f"{tokens[i]} {tokens[i + 1]}"
            if bi in phrase_keys:
                word_score = get_token_score(bi)
                matched_len = 2
                matched_phrase = bi

        # Cek unigram (1 kata) jika bigram dan trigram tidak cocok
        if word_score == 0.0:
            word_score = get_token_score(tokens[i])
            matched_phrase = tokens[i]

        # Jika kata ditemukan di kamus, proses skornya
        if word_score != 0.0:
            # Ambil window (kata-kata sebelum token saat ini)
            window_left = tokens[max(0, i - window_size): i]

            # Cari kata negasi terdekat dari belakang (paling nempel ke kata yang dievaluasi)
            negation_word = next((w for w in reversed(window_left) if w in negation_words), None)

            if negation_word:
                word_score = -word_score
                # Menggabungkan kata negasi asli dengan frasa yang terdeteksi
                matched_phrase = f"{negation_word} {matched_phrase}"

            score += word_score
            token_scores[matched_phrase] = word_score # Merekam skor token final

        i += matched_len

    # 5. Aturan Penentuan Label Akhir
    if score > 1:
        sentimen = "Positive"
    elif score < -1:
        sentimen = "Negative"
    else:
        sentimen = "Neutral"

    return score, sentimen, str(token_scores)

# Fungsi apply akan memecah 3 output fungsi dan langsung memperbarui/menambahkannya ke df_inset
data[['score', 'label', 'token_score']] = data['text_filtered'].apply(
    lambda x: pd.Series(sentiment_analysis_lexicon_indonesia(x))
)

# Mengecek hasil dari 5 baris pertama untuk memastikan kolom token_score terisi dengan benar
print(data[['text', 'score', 'label', 'token_score']].head())

                                                text  score     label  \
0  Polisi tenan i kon niru kinerja ne polisi sken...    8.0  Positive   
3  Semoga aplikasi pengaduan ini bisa menyempurna...   -6.0  Negative   
5  @IUlilyy @bening_93 Semua orang yang pernah ke...   -9.0  Negative   
6  @septjunior Kenapa polisi kita beda sm yg di 8...   -2.0  Negative   
7  setelah semua gak diterima beneran gak mau per...  -16.0  Negative   

                                         token_score  
0                      {'meniru': 4.0, 'lugas': 4.0}  
3  {'aplikasi': -4.0, 'menyempurnakan': 3.0, 'kem...  
5  {'kehilangan': -4.0, 'lapor': 2.0, 'enggak wkw...  
6                                     {'beda': -2.0}  
7  {'enggak diterima': -2.0, 'enggak percaya': -2...  


In [ ]:
data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 5457 entries, 0 to 6721
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   text            5457 non-null   object 
 1   created_at      5457 non-null   object 
 2   id_str          5457 non-null   int64  
 3   lang            5457 non-null   object 
 4   text_casefold   5457 non-null   object 
 5   text_char       5457 non-null   object 
 6   text_token      5457 non-null   object 
 7   english_tokens  5457 non-null   object 
 8   text_norm       5457 non-null   object 
 9   text_filtered   5457 non-null   object 
 10  score           5457 non-null   float64
 11  label           5457 non-null   object 
 12  token_score     5457 non-null   object 
dtypes: float64(1), int64(1), object(11)
memory usage: 725.9+ KB


In [ ]:
inset_counts = data['label'].value_counts()
inset_counts

,count
label,
Negative,3037
Positive,1445
Neutral,975


In [ ]:
# 1. Mendefinisikan kolom apa saja yang ingin diambil
kolom_pilihan = ['text', 'created_at', 'id_str', 'text_filtered', 'score', 'label', 'token_score']

# 2. Membuat DataFrame baru yang hanya berisi kolom-kolom tersebut
df_simpan = data[kolom_pilihan]

# 3. Menentukan lokasi penyimpanan (bisa disesuaikan nama filenya)
path_simpan = '/content/drive/MyDrive/Tugas Akhir/inset/data_inset.xlsx'

# 4. Menyimpan data ke format Excel tanpa menyertakan index baris
df_simpan.to_excel(path_simpan, index=False)

print(f"Berhasil menyimpan {len(df_simpan)} baris data ke: {path_simpan}")

Berhasil menyimpan 5457 baris data ke: /content/drive/MyDrive/Tugas Akhir/inset/data_inset.xlsx


In [ ]:
df1 = pd.read_excel("/content/drive/MyDrive/Tugas Akhir/inset/data_inset.xlsx")
df1.head()

,text,created_at,id_str,text_filtered,score,label,token_score
0,Polisi tenan i kon niru kinerja ne polisi sken...,2023-07-04,1676278088077046016,"['polisi', 'meniru', 'kinerja', 'polisi', 'ske...",8,Positive,"{'meniru': 4.0, 'lugas': 4.0}"
1,Semoga aplikasi pengaduan ini bisa menyempurna...,2023-07-04,1676176799817170944,"['semoga', 'aplikasi', 'pengaduan', 'menyempur...",-6,Negative,"{'aplikasi': -4.0, 'menyempurnakan': 3.0, 'kem..."
2,@IUlilyy @bening_93 Semua orang yang pernah ke...,2023-07-04,1676167467327160064,"['orang', 'kehilangan', 'motor', 'enggak', 'me...",-9,Negative,"{'kehilangan': -4.0, 'lapor': 2.0, 'enggak wkw..."
3,@septjunior Kenapa polisi kita beda sm yg di 8...,2023-07-04,1676153500517150976,"['polisi', 'beda', 'panggilannya', 'kinerja', ...",-2,Negative,{'beda': -2.0}
4,setelah semua gak diterima beneran gak mau per...,2023-07-04,1676117351270600960,"['enggak', 'diterima', 'benaran', 'enggak', 'p...",-16,Negative,"{'enggak diterima': -2.0, 'enggak percaya': -2..."


In [ ]:
df1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5457 entries, 0 to 5456
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   text           5457 non-null   object
 1   created_at     5457 non-null   object
 2   id_str         5457 non-null   int64 
 3   text_filtered  5457 non-null   object
 4   score          5457 non-null   int64 
 5   label          5457 non-null   object
 6   token_score    5457 non-null   object
dtypes: int64(2), object(5)
memory usage: 298.6+ KB
